# 02 — LRP for Text Tokens (interactive)

Computes input-embedding gradient relevance over the prompt for `text` and `image+text` modes.
Two passes: forward on cached prediction tokens, then `.backward()` on predicted-label logits.

**No generation** — loads the cache from `01_inference.ipynb`.

For grid runs across models and datasets use `run_lrp_text.py` and `run_all.sh`.
Outputs: per-sample `{token_ids, relevance_scores}` → `data/lrp_results/text_<mode>/`


In [1]:
CONFIG = {
    "dataset_path": "../../datasets/translated/",
    "language": "uk",
    "model_id": "Qwen/Qwen3-VL-4B-Instruct",
    "inference_cache_dir": "./data/inference_cache",
    "output_dir": "./data/lrp_results",
    "modes": ["text", "image+text"],
    "seed": 42,
}

In [2]:
import json
import re
import traceback
from functools import partial
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.nn import Dropout, LayerNorm
from tqdm import tqdm
from transformers import AutoProcessor
from transformers.models.qwen3_vl import modeling_qwen3_vl
from transformers.models.qwen3_vl.modeling_qwen3_vl import Qwen3VLTextMLP, Qwen3VLTextRMSNorm

from lxt.efficient import monkey_patch
from lxt.efficient.patches import (
    dropout_forward,
    gated_mlp_forward,
    layer_norm_forward,
    patch_attention,
    patch_method,
    rms_norm_forward,
)
from lxt.efficient.zennit_patches import monkey_patch_zennit

OUT_DIR = Path(CONFIG["output_dir"])
CACHE_DIR = Path(CONFIG["inference_cache_dir"])

Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


In [3]:
attn_lrp = {
    Qwen3VLTextMLP: partial(patch_method, gated_mlp_forward),
    Qwen3VLTextRMSNorm: partial(patch_method, rms_norm_forward),
    Dropout: partial(patch_method, dropout_forward),
    modeling_qwen3_vl: patch_attention,
    LayerNorm: partial(patch_method, layer_norm_forward),
}
monkey_patch(modeling_qwen3_vl, patch_map=attn_lrp, verbose=True)
monkey_patch_zennit(verbose=True)

Patched Qwen3VLTextMLP
Patched Qwen3VLTextRMSNorm
Patched Dropout
Patching attention function: flash_attention_3
Patching attention function: flash_attention_2
Patching attention function: flex_attention
Patching attention function: paged_attention
Patching attention function: sdpa
Patching attention function: sdpa_paged
Patching attention function: eager_paged
Patched transformers.models.qwen3_vl.modeling_qwen3_vl
Patched LayerNorm
Patched Zennit BasicHook's forward
Patched Zennit BasicHook's backward


In [4]:
model = modeling_qwen3_vl.Qwen3VLForConditionalGeneration.from_pretrained(
    CONFIG["model_id"], device_map="cuda", dtype=torch.bfloat16,
)
model.eval()
processor = AutoProcessor.from_pretrained(CONFIG["model_id"])
print(f"Model loaded: {CONFIG['model_id']}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen3-VL-4B-Instruct


In [5]:
def find_subsequence(sequence, subsequence):
    if not subsequence or len(subsequence) > len(sequence):
        return -1
    w = len(subsequence)
    for i in range(len(sequence) - w + 1):
        if sequence[i:i+w] == subsequence:
            return i
    return -1

def normalize_relevance(rel):
    rel = rel.clone().float()
    rel = torch.nan_to_num(rel, nan=0.0, posinf=0.0, neginf=0.0)
    return rel / (rel.abs().max() + 1e-12)

def find_label_positions(seq, target_labels, input_len):
    positions = []
    for label in target_labels:
        label_ids = processor.tokenizer.encode(label, add_special_tokens=False)
        for idx in range(input_len, len(seq) - len(label_ids) + 1):
            if seq[idx:idx+len(label_ids)] == label_ids:
                positions.extend(range(idx, idx+len(label_ids)))
                break
    return sorted(set(positions))

def compute_text_lrp(record, mode):
    """Compute gradient×input relevance over the OCR text span in the prompt."""
    output_ids = torch.tensor(record["output_ids"], dtype=torch.long).unsqueeze(0).to("cuda")
    input_ids = torch.tensor(record["input_ids"], dtype=torch.long).unsqueeze(0).to("cuda")
    input_len = record["input_len"]
    pred_labels = record.get("pred_labels", [])

    if not pred_labels:
        return None

    seq = record["output_ids"]
    label_positions = find_label_positions(seq, pred_labels, input_len)
    if not label_positions:
        return None

    logit_positions = [p - 1 for p in label_positions if p > 0]
    target_token_ids = [seq[p] for p in label_positions if p > 0]

    input_embeds = model.get_input_embeddings()(output_ids).detach().requires_grad_(True)

    model_kwargs = {
        "inputs_embeds": input_embeds,
        "attention_mask": torch.ones_like(output_ids),
        "use_cache": False,
    }
    if mode == "image+text" and record.get("image_grid_thw") is not None:
        # Re-load pixel values for image+text mode
        sample_id = record["id"]
        from pathlib import Path as P
        dataset_path = P(CONFIG["dataset_path"])
        # We'll pass image_grid_thw; pixel_values already encoded in input_ids via vision tokens
        # For text LRP we only need gradients over token embeddings, so no pixel_values needed
        pass

    outputs = model(**model_kwargs)
    logits = outputs.logits
    target_logits = logits[0, logit_positions, target_token_ids]
    target_logits.sum().backward()

    relevance = (input_embeds * input_embeds.grad).float().sum(-1).detach().cpu()[0]
    relevance_norm = normalize_relevance(relevance)

    return {
        "token_ids": record["output_ids"],
        "input_len": input_len,
        "relevance_raw": relevance.tolist(),
        "relevance_norm": relevance_norm.tolist(),
        "label_positions": label_positions,
    }

In [6]:
def run_lrp_text(mode):
    mode_key = mode.replace("+", "_")
    cache_dir = CACHE_DIR / mode_key
    out_dir = OUT_DIR / f"text_{mode_key}"
    out_dir.mkdir(parents=True, exist_ok=True)

    manifest = json.load(open(cache_dir / "manifest.json"))
    failures = []

    for entry in tqdm(manifest, desc=f"LRP text [{mode}]"):
        sample_id = entry["id"]
        out_file = out_dir / f"{sample_id}.json"
        if out_file.exists():
            continue

        try:
            record = json.load(open(entry["path"]))
            lrp_result = compute_text_lrp(record, mode)

            if lrp_result is None:
                failures.append({"id": sample_id, "error": "no labels or alignment"})
                continue

            lrp_result["id"] = sample_id
            lrp_result["mode"] = mode
            with open(out_file, "w", encoding="utf-8") as f:
                json.dump(lrp_result, f, ensure_ascii=False, indent=2)

        except Exception as e:
            print(f"  FAIL {sample_id}: {e}")
            traceback.print_exc()
            failures.append({"id": sample_id, "error": str(e)})
        finally:
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()

    with open(out_dir / "failures.json", "w") as f:
        json.dump(failures, f, indent=2)

    n_ok = len(list(out_dir.glob("*.json"))) - 1  # exclude failures.json
    print(f"Mode '{mode}': {n_ok} saved, {len(failures)} failures")

In [7]:
for mode in CONFIG["modes"]:
    run_lrp_text(mode)
print("\nText LRP done.")

LRP text [text]: 100%|██████████| 158/158 [01:25<00:00,  1.85it/s]


Mode 'text': 158 saved, 0 failures


LRP text [image+text]: 100%|██████████| 158/158 [01:35<00:00,  1.65it/s]

Mode 'image+text': 158 saved, 0 failures

Text LRP done.
